**TMDB Movie Data Analysis**

In [1]:
import pandas as pd

In [2]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip  install requests

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MovieDataAnalysis") \
    .config("spark.sql.warehouse.dir", "file:///C:/tmp") \
    .config("spark.hadoop.tmp.dir", "file:///C:/tmp") \
    .config("spark.local.dir", "C:/Users/HP/Desktop/Data_Engineer-Amalitech/LABS/PYSPARK/PySpark_Movie_Data_Analysis/temp") \
    .getOrCreate()


In [5]:
!pip install dotenv
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, BooleanType, FloatType
import os

In [6]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

True

In [7]:
api_key = os.getenv("TMDB_API_KEY")
if not api_key:
    raise ValueError("TMDB_API_KEY environment variable not set.")

In [8]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from utils import extract_movie_data, clean_movie_data, analyze_movie_data, visualize_movie_data

In [9]:
movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397, 420818, 24428, 168259, 99861, 284054, 12445, 181808, 330457, 351286, 109445, 321612, 260513]

In [10]:
from utils import extract_tmdb_data

In [11]:
# Extract the movie data using the function from utils.py
movie_data_df = extract_tmdb_data(movie_ids, api_key)

Error fetching data for movie ID 0: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/0?api_key=356d9ae6fc108307ae1b5f6358da93ee
Movie ID 0 not found. Skipping.


In [12]:
movie_data_df.show(5, vertical=True)

-RECORD 0-------------------------------------
 adult                 | false                
 backdrop_path         | /7RyHsO4yDXtBv1zU... 
 belongs_to_collection | {backdrop_path ->... 
 budget                | 356000000            
 genres                | [{name -> NULL, i... 
 homepage              | https://www.marve... 
 id                    | 299534               
 imdb_id               | tt4154796            
 origin_country        | [US]                 
 original_language     | en                   
 original_title        | Avengers: Endgame    
 overview              | After the devasta... 
 popularity            | 79.3322              
 poster_path           | /ulzhLuWrPK07P1Yk... 
 production_companies  | [{name -> NULL, i... 
 production_countries  | [{name -> United ... 
 release_date          | 2019-04-24           
 revenue               | 2799439100           
 runtime               | 181                  
 spoken_languages      | [{name -> English... 
 status      

In [13]:
# Check the schema of the DataFrame
movie_data_df.printSchema()

root
 |-- adult: boolean (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)
 |-- budget: long (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull = true)
 |-- homepage: string (nullable = true)
 |-- id: long (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- origin_country: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- production_companies: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull

In [14]:
from utils import flatten_df

flattened_df = flatten_df(movie_data_df)

In [15]:
# Use Parquet (efficient for Spark)
flattened_df.write.mode("overwrite").parquet("file:///C:/Users/HP/Desktop/PySpark_Results/flattened_movie_data.parquet")

Py4JJavaError: An error occurred while calling o86.parquet.
: java.lang.UnsatisfiedLinkError: org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Ljava/lang/String;I)Z
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(FileOutputComm

In [21]:
flattened_df.printSchema()

root
 |-- adult: boolean (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)
 |-- budget: long (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull = true)
 |-- homepage: string (nullable = true)
 |-- id: long (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- origin_country: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- production_companies: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull

In [ ]:
# Show schema
loaded_movie_data_df.printSchema()

In [ ]:
# Quick check of data
loaded_movie_data_df.show()

Step 2: Data Cleaning and Preprocessing (PySpark)

In [ ]:
# Clean the data
cleaned_movie_data_df = clean_movie_data(loaded_movie_data_df)

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "from_json(genres)" due to data type mismatch: Parameter 1 requires the "STRING" type, however "genres" has the type "ARRAY<STRUCT<id: INT, name: STRING>>".;
'Project [budget#7, genres#8, id#9, original_language#10, overview#50, popularity#13, poster_path#14, release_date#15, revenue#16L, runtime#17, status#18, tagline#19, title#73, vote_average#22, vote_count#23, belongs_to_collection#24, production_companies#25, production_countries#26, spoken_languages#27, from_json(ArrayType(StructType(StructField(id,IntegerType,true),StructField(name,StringType,true)),true), genres#8, Some(GMT)) AS genres_parsed#115]
+- Project [budget#7, genres#8, id#9, original_language#10, overview#50, popularity#13, poster_path#14, release_date#15, revenue#16L, runtime#17, status#18, tagline#19, title#73, vote_average#22, vote_count#23, belongs_to_collection#24, production_companies#25, production_countries#26, spoken_languages#27]
   

In [ ]:
# Check cleaned data
cleaned_movie_data_df.show()

Step 3: Analyze Data

In [ ]:
# Analyze movie data (KPIs)
analysis_results = analyze_movie_data(cleaned_movie_data_df)

Step 4: Visualize Data

In [ ]:
# Visualize movie data
visualize_movie_data(cleaned_movie_data_df, analysis_results)

In [ ]:
# Print some results to the console
print("Highest Revenue Movies:")
analysis_results["highest_revenue_movies"].show()

In [ ]:
print("\nMost Popular Movies:")
analysis_results["most_popular_movies"].show()

In [ ]:
print("\nFranchise vs Standalone Performance")
analysis_results["franchise_performance_df"].show()

In [ ]:
# Stop the Spark session
spark.stop()

In [ ]:
if __name__ == "__main__":
    main()